In [5]:
# =============================================================================
# RANDOM FOREST BIAS CORRECTION OF THE ENSEMBLE: TRAINING vs TESTING, RAW vs CORRECTED
# =============================================================================
# PURPOSE (reviewer comments 2 and 3):
#   - Rainfall and temperature come from the SAME five-model ensemble, so no
#     rainfall from one model is paired with temperature from another.
#   - The RF set-up is written out in full (Table 1): target, predictors,
#     settings, spatial and seasonal treatment, and the training and testing
#     samples.
#   - The RF is built on 1981-2000 (training) and tested on 2001-2014 (testing),
#     so the testing years are not used to build the correction.
#   - Raw and corrected ensembles are both compared with observations.
#
# RF SET-UP:
#   Target     : difference = observed - raw ensemble (mm/month, degC)
#   Predictors : raw ensemble value, month (sin and cos, so December is next to
#                January), longitude, latitude
#   Corrected  : raw ensemble + predicted difference (rainfall is kept >= 0)
#   Samples    : every basin pixel in every month, pooled
#   Observed   : CHIRPS (rainfall), ERA5-Land (temperature), same 0.25 degree grid
#
# FIGURES (panel letters a, b, c ... at the top centre of each panel):
#   Figure_1_scatter.png        raw and corrected vs observed, training and testing
#   Figure_2_seasonal.png       monthly average cycle, testing years
#   Figure_3_bias_maps.png      mean bias per pixel, raw and corrected, testing years
#   Figure_4_variation_link.png month-to-month variation and rainfall-temperature link
#   Figure_5_scenarios.png      yearly basin average, raw and corrected, both scenarios
# TABLES (CSV):
#   Table1_RF_setup.csv, Table2_performance.csv, Table3_variation_link.csv,
#   Table4_scenarios.csv, Seasonal_cycle.csv
#
# INPUT:
#   CHIRPS_resampled\yyyy_mm.tif, ERA5_resampled\yyyy_mm.tif
#   NEX-GDDP-CMIP6\{pr,ta}\ENSEMBLE\{historical,ssp245,ssp585}\yyyy_mm.tif   (raw)
#   NEX-GDDP-CMIP6\{pr,ta}_corrected_RF\ENSEMBLE\{...}\yyyy_mm.tif            (corrected,
#       final RF fitted on 1981-2014, used for the projections)
#   NEX-GDDP-CMIP6\{pr,ta}\{model}\historical\yyyy_mm.tif  (single models, for Figure 4)
# OUTPUT: NEX-GDDP-CMIP6\RF_evaluation_final\
# =============================================================================

import os
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =============================================================================
# SETTINGS AND PATHS
# =============================================================================

BASE_DIR = r"C:\Users\gerii\Documents\projects\tekeze"
NEX_DIR  = os.path.join(BASE_DIR, "NEX-GDDP-CMIP6")
OBS_DIRS = {"pr": os.path.join(BASE_DIR, "CHIRPS_resampled"),
            "ta": os.path.join(BASE_DIR, "ERA5_resampled")}
RAW_ENS  = {v: os.path.join(NEX_DIR, v, "ENSEMBLE") for v in ("pr", "ta")}
COR_ENS  = {v: os.path.join(NEX_DIR, f"{v}_corrected_RF", "ENSEMBLE") for v in ("pr", "ta")}
OUT_DIR  = os.path.join(NEX_DIR, "RF_evaluation_final")
os.makedirs(OUT_DIR, exist_ok=True)

MODELS = ["EC-Earth3", "GFDL-ESM4", "GFDL-CM4", "NorESM2-MM", "INM-CM5-0"]
TRAIN_Y = (1981, 2000)
TEST_Y  = (2001, 2014)
SCEN = {"ssp245": (2030, 2050), "ssp585": (2030, 2050)}

SEED = 42
RF_PARAMS = dict(n_estimators=200, min_samples_leaf=5, max_features=0.6,
                 n_jobs=-1, random_state=SEED)
MAX_PLOT = 40000
rng = np.random.default_rng(SEED)

VAR = {"pr": {"name": "Rainfall", "unit": "mm/month", "obs": "CHIRPS"},
       "ta": {"name": "Air temperature", "unit": "°C", "obs": "ERA5-Land"}}

FS_LABEL, FS_TICK, FS_STATS, FS_PANEL = 17, 18, 14, 24
plt.rcParams.update({"font.size": FS_LABEL, "axes.labelsize": FS_LABEL,
                     "xtick.labelsize": FS_TICK, "ytick.labelsize": FS_TICK,
                     "legend.fontsize": FS_TICK - 1, "font.family": "serif",
                     "axes.linewidth": 1.3})

def letter(ax, t):
    ax.text(0.5, 1.02, t, transform=ax.transAxes, fontsize=FS_PANEL,
            fontweight="bold", ha="center", va="bottom")

def keys(y0, y1):
    return [f"{y}_{m:02d}" for y in range(y0, y1 + 1) for m in range(1, 13)]

def load(path, var):
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as s:
        a = s.read(1).astype(np.float32)
        nd = s.nodata
    if nd is not None and not np.isnan(nd):
        a[a == nd] = np.nan
    a[a <= -9999] = np.nan
    if var == "ta" and np.isfinite(a).any() and np.nanmean(a) > 100:
        a = a - 273.15                      # Kelvin to degC
    return a

def load_series(folder, var, ks):
    d = {}
    for k in ks:
        a = load(os.path.join(folder, f"{k}.tif"), var)
        if a is not None:
            d[k] = a
    return d

# =============================================================================
# LOAD
# =============================================================================

print("=" * 70)
print("RF BIAS CORRECTION: TRAINING vs TESTING, RAW vs CORRECTED")
print("=" * 70)

hist_keys  = keys(1981, 2014)
train_keys = keys(*TRAIN_Y)
test_keys  = keys(*TEST_Y)

raw, obs = {}, {}
for v in ("pr", "ta"):
    raw[v] = load_series(os.path.join(RAW_ENS[v], "historical"), v, hist_keys)
    obs[v] = load_series(OBS_DIRS[v], v, hist_keys)
    print(f"  {v.upper()}: raw ensemble months = {len(raw[v])}, observed months = {len(obs[v])}")
if not raw["pr"] or not obs["pr"]:
    raise SystemExit("Raw ensemble or observed files not found. Check the folders.")

ref_key = next(iter(raw["pr"]))
with rasterio.open(os.path.join(RAW_ENS["pr"], "historical", f"{ref_key}.tif")) as r:
    tr, bounds = r.transform, r.bounds
H, W = raw["pr"][ref_key].shape
rows_i, cols_i = np.indices((H, W))
LON = (tr.c + (cols_i + 0.5) * tr.a).astype(np.float32)
LAT = (tr.f + (rows_i + 0.5) * tr.e).astype(np.float32)
extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)

# basin pixels: valid in the raw ensemble and in the observations
basin = np.isfinite(raw["pr"][ref_key]) & np.isfinite(obs["pr"][ref_key])
print(f"  Basin pixels: {int(basin.sum())}")

def feats(a, month, mask):
    n = int(mask.sum())
    ang = 2 * np.pi * month / 12.0
    return np.column_stack([a[mask], np.full(n, np.sin(ang)), np.full(n, np.cos(ang)),
                            LON[mask], LAT[mask]]).astype(np.float32)

# =============================================================================
# BUILD THE RF (training years only) AND CORRECT ALL MONTHS
# =============================================================================

corr = {}
n_samples = {}
for v in ("pr", "ta"):
    Xs, ds = [], []
    for k in train_keys:
        if k in raw[v] and k in obs[v]:
            m = basin & np.isfinite(raw[v][k]) & np.isfinite(obs[v][k])
            Xs.append(feats(raw[v][k], int(k[5:7]), m))
            ds.append(obs[v][k][m] - raw[v][k][m])
    X, d = np.concatenate(Xs), np.concatenate(ds)
    n_samples[v] = len(d)
    rf = RandomForestRegressor(**RF_PARAMS).fit(X, d)
    print(f"  {v.upper()}: RF fitted on {len(d):,} training samples ({TRAIN_Y[0]}-{TRAIN_Y[1]})")

    corr[v] = {}
    for k, a in raw[v].items():
        m = basin & np.isfinite(a)
        out = np.full((H, W), np.nan, np.float32)
        out[m] = a[m] + rf.predict(feats(a, int(k[5:7]), m))
        if v == "pr":
            out = np.where(np.isfinite(out), np.maximum(out, 0), np.nan)
        corr[v][k] = out
    del rf

# =============================================================================
# POOLED SAMPLES AND PERFORMANCE
# =============================================================================

def pooled(v, ks, which):
    o_l, p_l = [], []
    for k in ks:
        if k in raw[v] and k in obs[v] and k in corr[v]:
            m = basin & np.isfinite(obs[v][k]) & np.isfinite(raw[v][k]) & np.isfinite(corr[v][k])
            o_l.append(obs[v][k][m])
            p_l.append((raw[v] if which == "Raw" else corr[v])[k][m])
    return np.concatenate(o_l), np.concatenate(p_l)

def stats(o, p):
    return {"R2": r2_score(o, p), "r": float(np.corrcoef(o, p)[0, 1]),
            "RMSE": float(np.sqrt(mean_squared_error(o, p))),
            "MAE": mean_absolute_error(o, p), "Bias": float(np.mean(p - o))}

PER = {"Training": train_keys, "Testing": test_keys}
perf, t2 = {}, []
for v in ("pr", "ta"):
    for per, ks in PER.items():
        for which in ("Raw", "Corrected"):
            o, p = pooled(v, ks, which)
            s = stats(o, p)
            perf[(v, per, which)] = (o, p, s)
            t2.append([VAR[v]["name"], per, which, len(o)] +
                      [round(s[c], 3) for c in ("R2", "r", "RMSE", "MAE", "Bias")])
T2 = pd.DataFrame(t2, columns=["Variable", "Period", "Series", "Samples", "R2", "r",
                               "RMSE", "MAE", "Bias"])
T2.to_csv(os.path.join(OUT_DIR, "Table2_performance.csv"), index=False, encoding="utf-8-sig")
print("\nTABLE 2: performance\n", T2.to_string(index=False))

# TABLE 1: RF set-up
n_pix = int(basin.sum())
T1 = pd.DataFrame([
    ["Method", "Random Forest regression (scikit-learn), one RF for rainfall and one for temperature"],
    ["Input data", "Equal-weight mean of 5 raw NEX-GDDP-CMIP6 models (same models for both variables)"],
    ["Observations", "CHIRPS (rainfall), ERA5-Land (temperature), same 0.25 degree grid"],
    ["Target", "Observed minus raw ensemble (mm/month for rainfall, degC for temperature)"],
    ["Corrected value", "Raw ensemble + predicted difference (rainfall kept >= 0)"],
    ["Predictors", "Raw ensemble value; month as sin and cos; longitude; latitude"],
    ["Seasonal treatment", "Month enters as sin and cos, so one RF covers all months"],
    ["Spatial treatment", f"{n_pix} basin pixels pooled; longitude and latitude as predictors"],
    ["Settings", "; ".join(f"{k} = {v}" for k, v in RF_PARAMS.items())],
    ["Training years", f"{TRAIN_Y[0]}-{TRAIN_Y[1]} ({n_samples['pr']:,} rainfall and {n_samples['ta']:,} temperature samples)"],
    ["Testing years", f"{TEST_Y[0]}-{TEST_Y[1]} (not used to build the RF)"],
    ["Final RF for projections", "Same set-up refitted on 1981-2014, then applied to ssp245 and ssp585"],
], columns=["Item", "Description"])
T1.to_csv(os.path.join(OUT_DIR, "Table1_RF_setup.csv"), index=False, encoding="utf-8-sig")

# =============================================================================
# FIGURE 1: SCATTER, RAW AND CORRECTED, TRAINING AND TESTING
# =============================================================================

order = [("Training", "Raw"), ("Training", "Corrected"), ("Testing", "Raw"), ("Testing", "Corrected")]
fig, axes = plt.subplots(2, 4, figsize=(28, 14.5))
fig.subplots_adjust(hspace=0.30, wspace=0.28, left=0.07, right=0.98, top=0.90, bottom=0.06)
cols_pt = {"pr": "#1f78b4", "ta": "#fdb863"}
n = 0
for i, v in enumerate(("pr", "ta")):
    for j, (per, which) in enumerate(order):
        ax = axes[i, j]
        o, p, s = perf[(v, per, which)]
        idx = rng.choice(len(o), min(MAX_PLOT, len(o)), replace=False)
        lo = np.percentile(np.concatenate([o[idx], p[idx]]), 0.5)
        hi = np.percentile(np.concatenate([o[idx], p[idx]]), 99.5)
        pad = 0.05 * (hi - lo); lo, hi = lo - pad, hi + pad
        ax.scatter(o[idx], p[idx], s=4, alpha=0.25, color=cols_pt[v], edgecolors="none", rasterized=True)
        ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.8)
        sl, ic = np.polyfit(o, p, 1)
        xf = np.linspace(lo, hi, 100)
        ax.plot(xf, sl * xf + ic, color="#d7191c", linewidth=2)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(f"Observed ({VAR[v]['unit']})", fontsize=21, fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"Ensemble ({VAR[v]['unit']})", fontsize=21, fontweight="bold")
        ax.text(0.05, 0.95, f"$R^2$ = {s['R2']:.2f}\nRMSE = {s['RMSE']:.2f}\nBias = {s['Bias']:.2f}",
                transform=ax.transAxes, fontsize=24, va="top",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="grey", alpha=0.85))
        ax.grid(True, linestyle="--", alpha=0.3)
        letter(ax, "abcdefgh"[n]); n += 1
for j, (per, which) in enumerate(order):
    fig.text(0.07 + (0.98 - 0.07) * (j + 0.5) / 4 - 0.01, 0.955,
             f"{per}, {which.lower()}", ha="center", fontsize=25, fontweight="bold")
fig.text(0.012, 0.72, "Rainfall (CHIRPS)", rotation=90, fontsize=25, fontweight="bold", va="center")
fig.text(0.012, 0.28, "Air temperature (ERA5-Land)", rotation=90, fontsize=25, fontweight="bold", va="center")
fig.savefig(os.path.join(OUT_DIR, "Figure_1_scatter.png"), dpi=450, bbox_inches="tight")
plt.close(fig)

# =============================================================================
# FIGURE 2: MONTHLY AVERAGE CYCLE (TESTING YEARS)
# =============================================================================

def basin_mean(d, k):
    a = d.get(k)
    return np.nan if a is None else float(np.nanmean(np.where(basin, a, np.nan)))

seas_rows = []
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, v, lt in zip(axes, ("pr", "ta"), "ab"):
    ser = {"Observed": obs[v], "Raw ensemble": raw[v], "Corrected ensemble": corr[v]}
    for name, d in ser.items():
        mon = [np.nanmean([basin_mean(d, k) for k in test_keys if int(k[5:7]) == m]) for m in range(1, 13)]
        for m, val in enumerate(mon, 1):
            seas_rows.append([VAR[v]["name"], name, m, round(float(val), 3)])
        st = {"Observed": ("black", "-"), "Raw ensemble": ("#2166ac", "--"),
              "Corrected ensemble": ("#d7191c", "-")}[name]
        ax.plot(range(1, 13), mon, color=st[0], linestyle=st[1], linewidth=2.5, marker="o", label=name)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    ax.set_xlabel("Month"); ax.set_ylabel(f"{VAR[v]['name']} ({VAR[v]['unit']})")
    ax.grid(True, linestyle="--", alpha=0.4); letter(ax, lt)
    if v == "pr":
        ax.legend(frameon=False)
fig.savefig(os.path.join(OUT_DIR, "Figure_2_seasonal.png"), dpi=400, bbox_inches="tight")
plt.close(fig)
pd.DataFrame(seas_rows, columns=["Variable", "Series", "Month", "Basin average"]).to_csv(
    os.path.join(OUT_DIR, "Seasonal_cycle.csv"), index=False, encoding="utf-8-sig")

# =============================================================================
# FIGURE 3: MEAN BIAS PER PIXEL, TESTING YEARS
# (colour map without white: dark blue - green - yellow - dark red;
#  area outside the basin in mid grey)
# =============================================================================

cmap_bias = copy.copy(plt.get_cmap("turbo"))
cmap_bias.set_bad("#808080")

bias_rows = []
fig, axes = plt.subplots(2, 2, figsize=(14, 13))
n = 0
for i, v in enumerate(("pr", "ta")):
    maps = {}
    for which, d in (("Raw", raw[v]), ("Corrected", corr[v])):
        st = [d[k] - obs[v][k] for k in test_keys if k in d and k in obs[v]]
        maps[which] = np.where(basin, np.nanmean(st, axis=0), np.nan)
        bias_rows.append([VAR[v]["name"], which, round(float(np.nanmean(maps[which])), 3),
                          round(float(np.nanmean(np.abs(maps[which]))), 3)])
    lim = float(np.nanmax(np.abs(maps["Raw"])))
    for j, which in enumerate(("Raw", "Corrected")):
        ax = axes[i, j]
        im = ax.imshow(maps[which], cmap=cmap_bias, vmin=-lim, vmax=lim, extent=extent,
                       interpolation="nearest", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        if i == 0:
            ax.set_title("")
        letter(ax, "abcd"[n]); n += 1
    cb = fig.colorbar(im, ax=axes[i, :], orientation="vertical", fraction=0.03, pad=0.03)
    cb.set_label(f"Mean bias ({VAR[v]['unit']})")
fig.text(0.30, 0.935, "Raw ensemble", ha="center", fontsize=FS_LABEL, fontweight="bold")
fig.text(0.62, 0.935, "Corrected ensemble", ha="center", fontsize=FS_LABEL, fontweight="bold")
fig.savefig(os.path.join(OUT_DIR, "Figure_3_bias_maps.png"), dpi=400, bbox_inches="tight")
plt.close(fig)
pd.DataFrame(bias_rows, columns=["Variable", "Series", "Mean bias", "Mean absolute bias"]).to_csv(
    os.path.join(OUT_DIR, "Pixel_bias_summary.csv"), index=False, encoding="utf-8-sig")

# =============================================================================
# TABLE 3 AND FIGURE 4: MONTH-TO-MONTH VARIATION AND RAINFALL-TEMPERATURE LINK
# (testing years; variation = departure of each month from the average of that
#  calendar month, taken on the basin-average series)
# =============================================================================

def series_of(d, ks):
    return np.array([basin_mean(d, k) for k in ks])

def dev(x):
    x = x.reshape(-1, 12)
    return (x - np.nanmean(x, axis=0)).ravel()

def link_stats(pr, ta):
    a, b = dev(pr), dev(ta)
    ok = np.isfinite(a) & np.isfinite(b)
    rho = spearmanr(a[ok], b[ok])[0]
    dry = a[ok] < np.percentile(a[ok], 25)
    hot = b[ok] > np.percentile(b[ok], 75)
    return float(np.std(a[ok])), float(np.std(b[ok])), float(rho), float(np.mean(dry & hot) * 100)

S = {"Observed": (series_of(obs["pr"], test_keys), series_of(obs["ta"], test_keys))}
members = {}
for mname in MODELS:
    p = load_series(os.path.join(NEX_DIR, "pr", mname, "historical"), "pr", test_keys)
    t = load_series(os.path.join(NEX_DIR, "ta", mname, "historical"), "ta", test_keys)
    if len(p) == len(test_keys) and len(t) == len(test_keys):
        members[mname] = (p, t)
    else:
        print(f"  WARNING: {mname} raw files incomplete for {TEST_Y}; left out of Figure 4")
for mname, (p, t) in members.items():
    S[f"{mname} (raw)"] = (series_of(p, test_keys), series_of(t, test_keys))
if "GFDL-CM4" in members and "INM-CM5-0" in members:
    S["Mixed pair: GFDL-CM4 rain + INM-CM5-0 temp. (raw)"] = (
        series_of(members["GFDL-CM4"][0], test_keys), series_of(members["INM-CM5-0"][1], test_keys))
S["Raw ensemble"] = (series_of(raw["pr"], test_keys), series_of(raw["ta"], test_keys))
S["Corrected ensemble"] = (series_of(corr["pr"], test_keys), series_of(corr["ta"], test_keys))

t3 = []
for name, (p, t) in S.items():
    sp, st, rho, comp = link_stats(p, t)
    t3.append([name, round(sp, 2), round(st, 2), round(rho, 2), round(comp, 1)])
T3 = pd.DataFrame(t3, columns=["Series", "Rainfall variation (SD, mm)", "Temperature variation (SD, degC)",
                               "Rainfall-temperature correlation", "Dry and hot months together (%)"])
T3.to_csv(os.path.join(OUT_DIR, "Table3_variation_link.csv"), index=False, encoding="utf-8-sig")
print("\nTABLE 3: variation and rainfall-temperature link (testing years)\n", T3.to_string(index=False))

colr = ["black"] + ["#9ecae1"] * (len(S) - 3) + ["#2166ac", "#d7191c"]
colr = colr[:len(S)]
fig, axes = plt.subplots(1, 3, figsize=(26, 9))
fig.subplots_adjust(bottom=0.36, wspace=0.28)
for ax, col, yl, lt in zip(axes, T3.columns[1:4],
                           ["Rainfall variation (SD, mm)", "Temperature variation (SD, °C)",
                            "Rainfall-temperature correlation"], "abc"):
    ax.bar(range(len(T3)), T3[col], color=colr, edgecolor="black")
    ax.set_xticks(range(len(T3)))
    ax.set_xticklabels(T3["Series"], rotation=60, ha="right", fontsize=FS_TICK - 3)
    ax.set_ylabel(yl); ax.axhline(0, color="black", linewidth=1)
    ax.grid(True, axis="y", linestyle="--", alpha=0.4); letter(ax, lt)
fig.savefig(os.path.join(OUT_DIR, "Figure_4_variation_link.png"), dpi=400, bbox_inches="tight")
plt.close(fig)

# =============================================================================
# SCENARIOS: RAW vs CORRECTED ENSEMBLE, ssp245 AND ssp585
# (corrected files are from the final RF fitted on 1981-2014)
# =============================================================================

scn = {}
for v in ("pr", "ta"):
    scn[v] = {"historical": (raw[v], load_series(os.path.join(COR_ENS[v], "historical"), v, hist_keys))}
    for sc, (y0, y1) in SCEN.items():
        ks = keys(y0, y1)
        scn[v][sc] = (load_series(os.path.join(RAW_ENS[v], sc), v, ks),
                      load_series(os.path.join(COR_ENS[v], sc), v, ks))

def mean_all(d):
    vals = [basin_mean(d, k) for k in d]
    return float(np.nanmean(vals)) if vals else np.nan

t4 = []
for v in ("pr", "ta"):
    rh, ch = scn[v]["historical"]
    for sc in SCEN:
        rf_, cf_ = scn[v][sc]
        if not rf_ or not cf_:
            continue
        t4.append([VAR[v]["name"], sc, round(mean_all(obs[v]) if False else mean_all(rh), 2),
                   round(mean_all(rf_), 2), round(mean_all(rf_) - mean_all(rh), 2),
                   round(mean_all(ch), 2), round(mean_all(cf_), 2),
                   round(mean_all(cf_) - mean_all(ch), 2)])
T4 = pd.DataFrame(t4, columns=["Variable", "Scenario", "Raw 1981-2014", "Raw 2030-2050", "Raw change",
                               "Corrected 1981-2014", "Corrected 2030-2050", "Corrected change"])
T4.to_csv(os.path.join(OUT_DIR, "Table4_scenarios.csv"), index=False, encoding="utf-8-sig")
print("\nTABLE 4: raw and corrected change, 1981-2014 to 2030-2050\n", T4.to_string(index=False))

# FIGURE 5: yearly basin average, raw and corrected, both scenarios
def yearly(d):
    ys = sorted({int(k[:4]) for k in d})
    out = {y: np.nanmean([basin_mean(d, k) for k in d if int(k[:4]) == y]) for y in ys
           if sum(1 for k in d if int(k[:4]) == y) == 12}
    return list(out), list(out.values())

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
scol = {"ssp245": "#2166ac", "ssp585": "#d7191c"}
for ax, v, lt in zip(axes, ("pr", "ta"), "ab"):
    ox, oy = yearly(obs[v])
    ax.plot(ox, oy, color="black", linewidth=2.2, label="Observed")
    hx, hy = yearly(scn[v]["historical"][0]); ax.plot(hx, hy, color="grey", linestyle="--", linewidth=1.8, label="Raw ensemble, historical")
    hx, hy = yearly(scn[v]["historical"][1]); ax.plot(hx, hy, color="grey", linewidth=1.8, label="Corrected ensemble, historical")
    for sc in SCEN:
        rx, ry = yearly(scn[v][sc][0]); cx, cy = yearly(scn[v][sc][1])
        ax.plot(rx, ry, color=scol[sc], linestyle="--", linewidth=1.8, label=f"Raw, {sc.upper()}")
        ax.plot(cx, cy, color=scol[sc], linewidth=1.8, label=f"Corrected, {sc.upper()}")
    ax.set_xlabel("Year"); ax.set_ylabel(f"Yearly basin average {VAR[v]['name'].lower()} ({VAR[v]['unit']})")
    ax.grid(True, linestyle="--", alpha=0.4); letter(ax, lt)
    if v == "pr":
        ax.legend(frameon=False, fontsize=FS_TICK - 3, ncol=2)
fig.savefig(os.path.join(OUT_DIR, "Figure_5_scenarios.png"), dpi=400, bbox_inches="tight")
plt.close(fig)

print("\nSaved to:", OUT_DIR)
print("Finished Successfully.")

RF BIAS CORRECTION: TRAINING vs TESTING, RAW vs CORRECTED
  PR: raw ensemble months = 408, observed months = 408
  TA: raw ensemble months = 408, observed months = 408
  Basin pixels: 146
  PR: RF fitted on 35,040 training samples (1981-2000)
  TA: RF fitted on 35,040 training samples (1981-2000)

TABLE 2: performance
        Variable   Period    Series  Samples    R2     r   RMSE    MAE   Bias
       Rainfall Training       Raw    35040 0.699 0.846 55.866 32.601 -3.396
       Rainfall Training Corrected    35040 0.928 0.963 27.378 15.450  0.051
       Rainfall  Testing       Raw    24528 0.758 0.880 52.406 31.308  2.760
       Rainfall  Testing Corrected    24528 0.920 0.960 30.237 18.357  0.878
Air temperature Training       Raw    35040 0.671 0.919  2.453  2.044  1.763
Air temperature Training Corrected    35040 0.972 0.986  0.713  0.540  0.000
Air temperature  Testing       Raw    24528 0.665 0.932  2.583  2.200  2.013
Air temperature  Testing Corrected    24528 0.964 0.983  0.850 